# Lab 05: GAN on MNIST

**Module 05** | Read `notes/05-gan-fundamentals.pdf` before running this notebook.

Train a minimal GAN on MNIST. Monitor generator and discriminator losses and inspect generated digit grids.

## How to use this notebook

1. **Run cells top to bottom** (Shift+Enter). Do not skip markdown cells; they explain the code.
2. **Read before you run.** Each section tells you what the next code block does, which terms mean, and what the printed output should look like.
3. **Use the comments in code.** Lines starting with `#` explain what each step does in plain language.
4. **Check the output.** After each code cell, compare your printed numbers to the "What to look for" notes.

Code is complete. You are not expected to fill in missing sections or debug skeleton code.


## Runtime device (CPU or GPU)

**What this section does:** PyTorch can run math on your **CPU** (the main processor) or on an **NVIDIA GPU** through **CUDA**. GPUs are faster for neural networks because they run many small matrix operations in parallel.

**Key terms:**
- **CPU:** Always available. Training works but can be slower on large models.
- **GPU / CUDA:** Optional hardware acceleration. If you have an NVIDIA GPU and drivers installed, PyTorch will use it automatically.
- **VRAM:** GPU memory. If you run out, reduce batch size.

The next cell prints which device is active so you know what to expect for training speed.


In [ ]:
import torch

# Pick GPU if CUDA is available; otherwise fall back to CPU (labs still run).
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

if device.type == "cuda":
    props = torch.cuda.get_device_properties(0)
    print(f"Using GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {props.total_memory / 1e9:.1f} GB")
else:
    print("Using CPU (no CUDA GPU detected).")
    print("All labs still run; training may take longer than on a GPU.")


## Key terms for this lab

| Term | Meaning |
|------|---------|
| **generator** | A network that maps random noise to synthetic images (fakes). |
| **discriminator** | A network that scores whether an image looks real or fake. |
| **latent vector** | A random noise vector (usually Gaussian) that seeds generation. |
| **adversarial training** | Two networks compete: D tries to catch fakes, G tries to fool D. |
| **BCE loss** | Binary Cross-Entropy: loss for yes/no (real vs fake) classification. |
| **detach** | Cut a tensor off from the computation graph so gradients do not flow through it. |

Refer back to this table whenever a term appears in code or output.


## What problem are we solving?

Most ML models learn from labeled examples (input -> correct output). **GANs** learn to **generate** new data without pixel-level targets. A **generator** creates fakes; a **discriminator** tries to tell real from fake. The generator improves only when it fools the discriminator.

## What you will learn

- How **adversarial training** works as a two-player game
- What a **latent vector** is and how the generator maps noise to images
- Why we use **BCE loss** for real/fake labels
- Why we **detach** fakes when training the discriminator
- How to read G and D losses plus D accuracy to judge training health

We use MNIST (28x28 handwritten digits) because it is small enough to train quickly on a laptop.


### Step 1: Load MNIST

**What this code does:** Downloads MNIST (if needed), normalizes pixels to [-1, 1], and builds a shuffled DataLoader.

**Why we do it:** The generator will use `Tanh` output in the same [-1, 1] range, so real and fake images live on the same scale.


**What to look for in the output**

About 60,000 train images, hundreds of batches per epoch, image shape (1, 28, 28), pixel range [-1.0, 1.0] after normalization.


In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
from torchvision.utils import make_grid

ROOT = Path("..").resolve()
DATA_DIR = ROOT / "datasets" / "mnist"
BATCH_SIZE = 128
LATENT_DIM = 100   # Size of the random noise vector fed to the generator
EPOCHS = 8
LR = 2e-4

# Scale pixels from [0,1] to [-1,1] to match generator Tanh output
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,)),
])

train_set = datasets.MNIST(root=str(DATA_DIR), train=True, download=True, transform=transform)
loader = DataLoader(train_set, batch_size=BATCH_SIZE, shuffle=True, drop_last=True)
print(f"MNIST train images: {len(train_set):,}")
print(f"Batches per epoch:  {len(loader)}")
print(f"Image shape:        {train_set[0][0].shape}  (channels, height, width)")
print(f"Pixel range:        [{train_set[0][0].min():.1f}, {train_set[0][0].max():.1f}]  (normalized)")


### Step 2: Visualize real training digits

**What this code does:** Shows a 4x4 grid of real MNIST digits from one training batch.

**Why we do it:** Before training anything, know what "real" looks like. The discriminator must learn this structure to reject blurry fakes.


**What to look for in the output**

A grid of recognizable handwritten digits. Labels printed below should be integers 0-9.


In [ ]:
real_batch, real_labels = next(iter(loader))
sample_grid = make_grid(real_batch[:16].cpu(), nrow=4, normalize=True, value_range=(-1, 1))

fig, ax = plt.subplots(figsize=(4, 4))
ax.imshow(sample_grid.permute(1, 2, 0).squeeze(), cmap="gray")
ax.set_title("16 real MNIST digits")
ax.axis("off")
plt.tight_layout()
plt.show()

print("Labels for the 16 samples above:", real_labels[:16].tolist())


### Step 3: Generator and discriminator

**What this code does:** Defines an MLP generator (noise -> 784 pixels -> 28x28 image) and an MLP discriminator (784 pixels -> real/fake probability). Creates optimizers for both.

**Why we do it:** These are the two players in the GAN game. The generator expands noise; the discriminator compresses an image to a single score.


**What to look for in the output**

Generator and discriminator parameter counts printed. Generator has more parameters because it expands noise to 784 pixels.


In [ ]:
class Generator(nn.Module):
    def __init__(self, latent_dim: int = 100):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(latent_dim, 256),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Linear(256, 512),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Linear(512, 1024),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Linear(1024, 28 * 28),
            nn.Tanh(),  # output pixels in [-1, 1]
        )

    def forward(self, z: torch.Tensor) -> torch.Tensor:
        return self.net(z).view(-1, 1, 28, 28)


class Discriminator(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Flatten(),
            nn.Linear(28 * 28, 512),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Linear(512, 256),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Linear(256, 1),
            nn.Sigmoid(),  # output in [0, 1] = probability of "real"
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.net(x)


G = Generator(LATENT_DIM).to(device)
D = Discriminator().to(device)
criterion = nn.BCELoss()  # binary cross-entropy for real (1) vs fake (0) labels
opt_g = torch.optim.Adam(G.parameters(), lr=LR, betas=(0.5, 0.999))
opt_d = torch.optim.Adam(D.parameters(), lr=LR, betas=(0.5, 0.999))
print(f"Generator params:     {sum(p.numel() for p in G.parameters()):,}")
print(f"Discriminator params: {sum(p.numel() for p in D.parameters()):,}")


**Concept: Adversarial training and detach**

Each training iteration has two phases:
1. **Train D:** Show real images (label 1) and fake images from G (label 0). Use `.detach()` on fakes so gradients update **only D**, not G.
2. **Train G:** Generate new fakes and label them as "real" (label 1). G receives a useful gradient only if it fools D.

This is **adversarial** because D and G have opposing goals. **BCE loss** measures how far D's outputs are from the desired 0/1 labels.


### Step 4: Adversarial training loop

**What this code does:** Alternates discriminator and generator updates for 8 epochs. Tracks both losses, D accuracy on real/fake batches, and mean D scores. Plots loss curves.

**Why we do it:** This is the core GAN training loop. Healthy training shows D separating real from fake while G gradually produces digit-like blobs.


**What to look for in the output**

Per-epoch G and D losses with D accuracy on real and fake. Mean D(real) should stay above D(fake). Losses should oscillate, not collapse to zero immediately.


In [ ]:
g_losses, d_losses = [], []
d_acc_real_hist, d_acc_fake_hist = [], []

for epoch in range(1, EPOCHS + 1):
    g_epoch, d_epoch = 0.0, 0.0
    d_correct_real, d_total_real = 0, 0
    d_correct_fake, d_total_fake = 0, 0
    last_d_real_mean, last_d_fake_mean = 0.0, 0.0

    for real, _ in loader:
        real = real.to(device)
        batch = real.size(0)
        real_labels = torch.ones(batch, 1, device=device)   # label 1 = real
        fake_labels = torch.zeros(batch, 1, device=device)   # label 0 = fake

        # --- Phase 1: update Discriminator ---
        D.zero_grad()
        d_real = D(real)
        loss_d_real = criterion(d_real, real_labels)
        z = torch.randn(batch, LATENT_DIM, device=device)
        fake = G(z).detach()  # detach: do not backprop into G while training D
        d_fake = D(fake)
        loss_d_fake = criterion(d_fake, fake_labels)
        loss_d = loss_d_real + loss_d_fake
        loss_d.backward()
        opt_d.step()

        d_correct_real += ((d_real > 0.5).float().sum().item())
        d_total_real += batch
        d_correct_fake += ((d_fake < 0.5).float().sum().item())
        d_total_fake += batch
        last_d_real_mean = d_real.mean().item()
        last_d_fake_mean = d_fake.mean().item()

        # --- Phase 2: update Generator ---
        G.zero_grad()
        z = torch.randn(batch, LATENT_DIM, device=device)
        fake = G(z)
        d_out = D(fake)
        # Trick: label fakes as "real" so G learns to fool D
        loss_g = criterion(d_out, real_labels)
        loss_g.backward()
        opt_g.step()

        g_epoch += loss_g.item()
        d_epoch += loss_d.item()

    g_losses.append(g_epoch / len(loader))
    d_losses.append(d_epoch / len(loader))
    acc_real = d_correct_real / d_total_real
    acc_fake = d_correct_fake / d_total_fake
    d_acc_real_hist.append(acc_real)
    d_acc_fake_hist.append(acc_fake)

    print(
        f"Epoch {epoch:02d}/{EPOCHS}  "
        f"G loss {g_losses[-1]:.4f}  D loss {d_losses[-1]:.4f}  "
        f"D acc real {acc_real * 100:.1f}%  fake {acc_fake * 100:.1f}%"
    )
    print(
        f"  Last-batch mean D(real)={last_d_real_mean:.3f}  "
        f"D(fake)={last_d_fake_mean:.3f}  "
        f"(want real high, fake low)"
    )

plt.figure(figsize=(7, 4))
plt.plot(range(1, EPOCHS + 1), g_losses, label="Generator", marker="o")
plt.plot(range(1, EPOCHS + 1), d_losses, label="Discriminator", marker="s")
plt.xlabel("Epoch")
plt.ylabel("BCE loss")
plt.title("GAN training losses")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()


### Step 5: Qualitative comparison of real vs. generated digits

**What this code does:** Shows side-by-side grids of 64 real MNIST digits and 64 freshly generated fakes.

**Why we do it:** Numbers alone can mislead. Your eyes catch whether G produces stroke-like blobs or pure noise.


**What to look for in the output**

Left grid: crisp real digits. Right grid: blurry but digit-shaped blobs (not uniform gray noise). Quality improves with more epochs.


In [ ]:
real_batch, _ = next(iter(loader))
real_batch = real_batch[:64].to(device)

G.eval()
with torch.no_grad():
    z = torch.randn(64, LATENT_DIM, device=device)
    fake_batch = G(z)

real_grid = make_grid(real_batch.cpu(), nrow=8, normalize=True, value_range=(-1, 1))
fake_grid = make_grid(fake_batch.cpu(), nrow=8, normalize=True, value_range=(-1, 1))

fig, axes = plt.subplots(1, 2, figsize=(10, 5))
axes[0].imshow(real_grid.permute(1, 2, 0).squeeze(), cmap="gray")
axes[0].set_title("Real MNIST")
axes[0].axis("off")
axes[1].imshow(fake_grid.permute(1, 2, 0).squeeze(), cmap="gray")
axes[1].set_title("Generated digits")
axes[1].axis("off")
plt.tight_layout()
plt.show()


### Step 6: Interpret the results

**What this code does:** Probes D on a batch of real and fake images and prints a plain-language interpretation of training health.

**Why we do it:** Ties together losses, accuracies, and the visual grid into actionable takeaways about whether the GAN is learning.


**What to look for in the output**

Mean D(real) noticeably above mean D(fake). Interpretation line should say the discriminator separates real and fake if training went well.


In [ ]:
with torch.no_grad():
    probe_real = next(iter(loader))[0][:128].to(device)
    probe_fake = G(torch.randn(128, LATENT_DIM, device=device))
    mean_real = D(probe_real).mean().item()
    mean_fake = D(probe_fake).mean().item()

print("=" * 60)
print("Lab 05 interpretation")
print("=" * 60)
print(f"  Final G loss:          {g_losses[-1]:.4f}")
print(f"  Final D loss:          {d_losses[-1]:.4f}")
print(f"  D accuracy on real:    {d_acc_real_hist[-1] * 100:.1f}%")
print(f"  D accuracy on fake:    {d_acc_fake_hist[-1] * 100:.1f}%")
print(f"  Mean D(real) probe:    {mean_real:.3f}  (closer to 1.0 is confident real)")
print(f"  Mean D(fake) probe:    {mean_fake:.3f}  (closer to 0.0 is confident fake)")
print()
if mean_real > mean_fake + 0.1:
    print("  Discriminator separates real and fake. Generator is learning.")
elif mean_real < 0.6 and mean_fake < 0.6:
    print("  Both scores are low. Discriminator may be winning too strongly.")
else:
    print("  Scores are close. Training may still be balancing.")
print("  Inspect the grid: digit-like blobs mean G is capturing stroke structure.")
print("=" * 60)
